In [1]:
import pandas as pd
import sys
import pickle
from sklearn.metrics import classification_report
from skopt.space import Real, Integer

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.helpers import Helpers

In [2]:
helper = Helpers()
properties = helper.load_properties()

try:
    feature_engineered_file_name = properties['LOCAL']['feature_engineered_file_name']
    RANDOM_STATE = properties['models']['random_state']
except KeyError as ke:
    raise KeyError(f"Missing key in properties file: {str(ke)}") from ke

In [3]:
# Apply global settings
helper.set_global_settings()

In [4]:
target_variable = 'loan_status'

parent_dir = helper.root_dir / "datasets"

train_df = pd.read_csv(parent_dir / "sampled_train_data.csv")
val_df = pd.read_csv(parent_dir / "validation_set.csv")
test_df = pd.read_csv(parent_dir / "test_set.csv")

X_train = train_df.drop(columns=[target_variable])
y_train = train_df[target_variable]

X_val = val_df.drop(columns=[target_variable])
y_val = val_df[target_variable]

X_test = test_df.drop(columns=[target_variable])
y_test = test_df[target_variable]

In [5]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_STATE)

search_space_xgb = {
    'learning_rate': Real(1e-3, 0.1, prior='log-uniform'),
    'n_estimators': Integer(100, 1000),
    'max_depth': Integer(3, 15),
    'subsample': Real(0.4, 1.0),
    'colsample_bytree': Real(0.4, 1.0),
    'gamma': Real(0, 5),
    'reg_alpha': Real(1e-3, 10, prior='log-uniform'),
    'reg_lambda': Real(1e-3, 10, prior='log-uniform')
}

opt_xgb_clf = helper.find_best_params(
    xgb_clf,
    search_space_xgb,
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    metric='f1_weighted'
    )

opt_xgb_clf.fit(X_train, y_train)
y_pred_xgb = opt_xgb_clf.predict(X_val)

Best Parameters:
{'colsample_bytree': 1.0,
 'gamma': 0.0,
 'learning_rate': 0.1,
 'max_depth': 11,
 'n_estimators': 1000,
 'reg_alpha': 0.001,
 'reg_lambda': 0.001,
 'subsample': 1.0}


In [6]:
# Define model
lgb_clf = LGBMClassifier(verbose=-1, objective='binary', metric='f1', random_state=RANDOM_STATE)

# Define search space
search_space_lgb = {
    'learning_rate': Real(1e-3, 0.1, prior='log-uniform'),
    'num_leaves': Integer(20, 150),
    'max_depth': Integer(3, 15),
    'min_child_samples': Integer(10, 200),
    'feature_fraction': Real(0.4, 1.0),
    'bagging_fraction': Real(0.4, 1.0),
    'bagging_freq': Integer(1, 7)
}

# Get best estimator using Bayesian Optimization
opt_lgb_clf = helper.find_best_params(
    lgb_clf, 
    search_space_lgb, 
    X_train,
    y_train,
    random_state=RANDOM_STATE, 
    metric='f1_weighted'
    )

opt_lgb_clf.fit(X_train, y_train)
y_pred = opt_lgb_clf.predict(X_val)

Best Parameters:
{'bagging_fraction': 0.7777346743917128,
 'bagging_freq': 1,
 'feature_fraction': 0.8296034164902779,
 'learning_rate': 0.1,
 'max_depth': 15,
 'min_child_samples': 10,
 'num_leaves': 150}


In [7]:
rf_clf = RandomForestClassifier(min_samples_leaf=50, max_features='sqrt', random_state=RANDOM_STATE)

search_space_rf = {
    'n_estimators': Integer(100, 1000),
    'max_depth': Integer(3, 15),
    'min_samples_split': Integer(2, 20),
}

opt_rf_clf = helper.find_best_params(
    rf_clf,
    search_space_rf,
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    metric='f1_weighted'
    )

opt_rf_clf.fit(X_train, y_train)
y_pred_rf = opt_rf_clf.predict(X_val)

Best Parameters:
{'max_depth': 15, 'min_samples_split': 20, 'n_estimators': 1000}


In [ ]:
bg_clf = BaggingClassifier(random_state=RANDOM_STATE)

search_space_bg = {
    'n_estimators': Integer(10, 200),
}

opt_bg_clf = helper.find_best_params(
    bg_clf,
    search_space_bg,
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    metric='f1_weighted'
    )

opt_bg_clf.fit(X_train, y_train)
y_pred_bg = opt_bg_clf.predict(X_val)

In [ ]:
et_clf = ExtraTreesClassifier(random_state=RANDOM_STATE)

search_space_et = {
    'n_estimators': Integer(100, 1000),
    'max_depth': Integer(3, 15),
    'min_samples_split': Integer(2, 20),
}

opt_et_clf = helper.find_best_params(
    et_clf,
    search_space_et,
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    metric='f1_weighted'
    )

opt_et_clf.fit(X_train, y_train)
y_pred_et = opt_et_clf.predict(X_val)

Best Parameters:
{'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 997}


In [ ]:
# Generate classification reports for all models with output_dict=True
reports = {
    'LightGBM': classification_report(y_val, opt_lgb_clf.predict(X_val), output_dict=True),
    'XGBoost': classification_report(y_val, opt_xgb_clf.predict(X_val), output_dict=True),
    'RandomForest': classification_report(y_val, opt_rf_clf.predict(X_val), output_dict=True),
    'ExtraTrees': classification_report(y_val, opt_et_clf.predict(X_val), output_dict=True),
    'Bagging': classification_report(y_val, opt_bg_clf.predict(X_val), output_dict=True),
}

average_type = 'weighted avg'

# Extract weighted metrics and create dataframe
metrics_df = pd.DataFrame({
    'Model': list(reports.keys()),
    'Weighted Precision': [reports[model][average_type]['precision'] for model in reports.keys()],
    'Weighted Recall': [reports[model][average_type]['recall'] for model in reports.keys()],
    'Weighted F1-Score': [reports[model][average_type]['f1-score'] for model in reports.keys()],
})

metrics_df.sort_values(by=['Weighted F1-Score', 'Weighted Recall', 'Weighted Precision'], ascending=False)

,Model,Weighted Precision,Weighted Recall,Weighted F1-Score
0,LightGBM,0.93,0.93,0.93
1,XGBoost,0.93,0.93,0.93
2,RandomForest,0.92,0.92,0.92
3,ExtraTrees,0.92,0.92,0.92
4,Bagging,0.92,0.92,0.92


In [12]:
# Evaluate best model on test set
y_test_pred = opt_lgb_clf.predict(X_test)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      3307
           1       0.88      0.81      0.84       947

    accuracy                           0.93      4254
   macro avg       0.91      0.89      0.90      4254
weighted avg       0.93      0.93      0.93      4254



In [ ]:
# Train best model on full data for deployment
best_model = LGBMClassifier(**opt_lgb_clf.get_params())
best_model.fit(X_train + X_val, y_train + y_val)

# Save the trained model
model_save_path = helper.root_dir / "models" / "best_model.pkl"
with open(model_save_path, 'wb') as f:
    pickle.dump(best_model, f)